# Data Acquisition for Electricity Price Forecasting

This notebook handles downloading and caching all required data for both DE-LU and ES zones.

## Data Sources
1. **Historical DAA Prices**: energy-charts.info
2. **Generation Data**: ENTSO-E Transparency Platform
3. **Weather Data**: Open-Meteo API / ERA5
4. **Fuel Prices**: ICE/EEX market data

## Time Range
- Training data: 2020-01-01 to 2026-05-05 (current date)
- Evaluation window: 2026-05-08 18:00 to 2026-05-09 23:00

In [ ]:
# Install dependencies if needed
%pip install -q pandas numpy requests tqdm

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
from tqdm import tqdm

from src.data.loaders import EnergyChartsLoader, WeatherDataLoader

print("Imports successful!")
print(f"Current date: {datetime.now()}")

## 1. Download Historical DAA Prices

In [ ]:
# Initialize loader
energy_loader = EnergyChartsLoader(cache_dir=Path('../data/raw'))

# Define date range
start_date = '2020-01-01'
end_date = '2026-05-05'

zones = ['DE-LU', 'ES']

# Download prices for both zones
prices_data = {}

for zone in zones:
    print(f"\n{'='*60}")
    print(f"Downloading DAA prices for {zone}")
    print(f"{'='*60}")
    
    df = energy_loader.load_day_ahead_prices(
        zone=zone,
        start_date=start_date,
        end_date=end_date,
        use_cache=False  # Set to True after first run
    )
    
    prices_data[zone] = df
    
    print(f"\nLoaded {len(df):,} hourly records")
    print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
    print(f"\nPrice statistics for {zone}:")
    print(df['price_eur_mwh'].describe())
    print(f"\nNegative prices: {(df['price_eur_mwh'] < 0).sum()} hours ({(df['price_eur_mwh'] < 0).mean()*100:.2f}%)")

## 2. Download Generation Data

In [ ]:
# Download generation data for both zones
generation_data = {}

for zone in zones:
    print(f"\n{'='*60}")
    print(f"Downloading generation data for {zone}")
    print(f"{'='*60}")
    
    df = energy_loader.load_generation_data(
        zone=zone,
        start_date=start_date,
        end_date=end_date,
        use_cache=False
    )
    
    generation_data[zone] = df
    
    print(f"\nLoaded {len(df):,} hourly records")
    print(f"\nGeneration mix statistics for {zone}:")
    print(df[['wind', 'solar', 'nuclear', 'gas', 'coal', 'hydro']].describe())

## 3. Download Weather Data

In [ ]:
# Initialize weather loader
weather_loader = WeatherDataLoader(cache_dir=Path('../data/external'))

# Download weather data for both zones
weather_data = {}

for zone in zones:
    print(f"\n{'='*60}")
    print(f"Downloading weather data for {zone}")
    print(f"{'='*60}")
    
    df = weather_loader.load_weather_data(
        zone=zone,
        start_date=start_date,
        end_date=end_date,
        use_cache=False
    )
    
    weather_data[zone] = df
    
    print(f"\nLoaded {len(df):,} hourly records")
    print(f"\nWeather statistics for {zone}:")
    print(df[['temperature_c', 'wind_speed_ms', 'solar_irradiance_wm2']].describe())

## 4. Data Quality Checks

In [ ]:
print("\n" + "="*60)
print("DATA QUALITY SUMMARY")
print("="*60)

for zone in zones:
    print(f"\n{zone}:")
    print("-" * 40)
    
    # Check for missing values
    print(f"Prices - Missing values: {prices_data[zone].isnull().sum().sum()}")
    print(f"Generation - Missing values: {generation_data[zone].isnull().sum().sum()}")
    print(f"Weather - Missing values: {weather_data[zone].isnull().sum().sum()}")
    
    # Check timestamp alignment
    print(f"\nTimestamp alignment:")
    print(f"  Prices: {len(prices_data[zone])} records")
    print(f"  Generation: {len(generation_data[zone])} records")
    print(f"  Weather: {len(weather_data[zone])} records")
    
    # Check for duplicates
    print(f"\nDuplicate timestamps:")
    print(f"  Prices: {prices_data[zone]['timestamp'].duplicated().sum()}")
    print(f"  Generation: {generation_data[zone]['timestamp'].duplicated().sum()}")
    print(f"  Weather: {weather_data[zone]['timestamp'].duplicated().sum()}")

## 5. Create Merged Dataset

In [ ]:
# Merge all data sources for each zone
merged_data = {}

for zone in zones:
    print(f"\nMerging data for {zone}...")
    
    # Start with prices
    df = prices_data[zone].copy()
    
    # Merge generation data
    df = df.merge(
        generation_data[zone],
        on='timestamp',
        how='left'
    )
    
    # Merge weather data
    df = df.merge(
        weather_data[zone],
        on='timestamp',
        how='left'
    )
    
    # Sort by timestamp
    df = df.sort_values('timestamp').reset_index(drop=True)
    
    merged_data[zone] = df
    
    # Save merged dataset
    output_path = Path(f'../data/processed/{zone.lower()}_merged.csv')
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    
    print(f"Saved merged dataset to {output_path}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")

## 6. Quick Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

# Plot price comparison
fig, axes = plt.subplots(2, 1, figsize=(15, 8))

for idx, zone in enumerate(zones):
    df = merged_data[zone]
    
    # Sample last 30 days for visualization
    df_recent = df.tail(30 * 24)
    
    axes[idx].plot(df_recent['timestamp'], df_recent['price_eur_mwh'], linewidth=0.8)
    axes[idx].set_title(f'{zone} - Day Ahead Prices (Last 30 Days)', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Price (EUR/MWh)')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].axhline(y=0, color='r', linestyle='--', alpha=0.5, label='Zero price')
    axes[idx].legend()

plt.tight_layout()
plt.savefig('../outputs/price_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nVisualization saved to outputs/price_comparison.png")

## Summary

Data acquisition complete! We now have:

1. ✅ Historical DAA prices (2020-2026) for DE-LU and ES
2. ✅ Generation data by source
3. ✅ Weather data (temperature, wind, solar)
4. ✅ Merged datasets ready for feature engineering

Next steps:
- Notebook 02: Exploratory Data Analysis
- Notebook 03: Feature Engineering
- Notebook 04: Model Development